# 03. Business Insights & Strategic Action Plan
### Track: Data Analytics (Executive Synthesis)

**Tujuan:**
1. Mengkristalisasi indikator kinerja utama (KPI) bisnis dan kuantifikasi dampak finansial churn.
2. Membandingkan ARPU (Average Revenue per User) antara pelanggan churn vs loyal.
3. Melakukan simulasi penghematan pendapatan bisnis (*Revenue Saved Simulation*).
4. Menyusun peta jalan aksi strategis (*Actionable Business Roadmap*) yang dapat dieksekusi tim bisnis/pemasaran.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "01_analytics" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.telco_churn.core.loader import load_processed_data
from src.telco_churn.analytics.metrics import calculate_churn_rate, calculate_mrr_loss, calculate_arpu

sns.set_theme(style="whitegrid", font="sans-serif")


In [ ]:
df = load_processed_data()
mrr_metrics = calculate_mrr_loss(df)
churn_rate = calculate_churn_rate(df)
arpu_overall = calculate_arpu(df)
arpu_churned = calculate_arpu(df[df["churn"] == "Yes"])
arpu_retained = calculate_arpu(df[df["churn"] == "No"])

print(f"=== RINGKASAN EKSEKUTIF KPI ===")
print(f"Total Pelanggan: {len(df):,}")
print(f"Baseline Churn Rate: {churn_rate:.2f}%")
print(f"Total MRR Perusahaan: ${mrr_metrics['total_mrr']:,.2f}")
print(f"Lost MRR akibat Churn: ${mrr_metrics['lost_mrr']:,.2f} ({mrr_metrics['pct_lost']:.2f}% dari total pendapatan)")
print(f"ARPU Keseluruhan: ${arpu_overall:.2f}")
print(f"ARPU Pelanggan Churn: ${arpu_churned:.2f} (vs Pelanggan Bertahan: ${arpu_retained:.2f})")


> **Temuan Kunci Finansial:**
> Pelanggan yang keluar (*churned*) memiliki rata-rata tagihan bulanan (**$74.44**) yang lebih tinggi **21.5%** dibandingkan pelanggan yang bertahan (**$61.27**). 
> Ini membuktikan bahwa churn tidak hanya menghilangkan volume pelanggan, tetapi secara sistematis mengikis segmen berpendapatan tinggi (*high-value customers*).


## 1. Analisis Peran Nilai Tambah Layanan (*VAS Protection Effect*)
Apakah fitur keamanan (`online_security`, `tech_support`, `online_backup`, `device_protection`) melindungi pelanggan dari churn?


In [ ]:
vas_cols = ["online_security", "tech_support", "online_backup", "device_protection"]
vas_data = []

for col in vas_cols:
    subset = df[df[col].isin(["Yes", "No"])]
    grp = subset.groupby(col)["churn"].apply(lambda s: (s == "Yes").mean() * 100)
    vas_data.append({
        "Layanan": col.replace("_", " ").title(),
        "Churn (%) Tanpa Layanan": grp.loc["No"],
        "Churn (%) Dengan Layanan": grp.loc["Yes"],
        "Penurunan Risiko": grp.loc["No"] - grp.loc["Yes"]
    })

vas_df = pd.DataFrame(vas_data)
vas_df


## 2. Simulasi Finansial Penghematan Churn (*Cost-Benefit Potential*)
Jika tim pemasaran dan retensi berhasil menurunkan churn rate sebesar 2% hingga 5% menggunakan intervensi berbasis data:


In [ ]:
scenarios = [0.01, 0.02, 0.03, 0.05]
sim_results = []

for sc in scenarios:
    saved_customers = int(len(df) * sc)
    monthly_savings = saved_customers * arpu_churned
    annual_savings = monthly_savings * 12
    sim_results.append({
        "Skenario Penurunan Churn": f"{int(sc*100)}%",
        "Pelanggan Terselamatkan": saved_customers,
        "Penghematan MRR / Bulan": f"${monthly_savings:,.2f}",
        "Penghematan ARR / Tahun": f"${annual_savings:,.2f}"
    })

pd.DataFrame(sim_results)


## 3. Matriks Rekomendasi Tindakan Strategis

| Prioritas | Pilar Strategi | Target Segmen | Rencana Aksi Bisnis | Proyeksi Dampak |
| :--- | :--- | :--- | :--- | :--- |
| **P1** | **Early Lifecycle Protection** | Pelanggan baru (Tenure 0-3 bulan) | • Onboarding VIP check-in di minggu ke-2.<br>• Transparansi simulasi tagihan pertama (*avoid bill shock*). | Menurunkan 35% churn lonjakan di bulan ke-1. |
| **P2** | **Contract Migration Campaign** | Pelanggan Month-to-Month dengan tenure > 6 bln | • Tawarkan voucher diskon $10/bulan untuk upgrade ke kontrak 1 tahun.<br>• Bundling gratis tech support selama 6 bulan. | Menurunkan churn segmen M2M dari 42.7% ke < 20%. |
| **P3** | **Digital Auto-Pay Shift** | Pengguna Electronic Check | • Program insentif reward cashback $5 satu kali untuk pendaftaran Credit Card / Bank Auto-Pay. | Memangkas friction pembayaran dan menurunkan churn dari 45.3% ke 16.7%. |
| **P4** | **Fiber Health & Stability** | Pengguna Fiber Optic ARPU tinggi | • Audit kualitas koneksi proaktif.<br>• Complimentary automated speed test dan prioritas customer service line. | Menyelamatkan segmen ARPU tinggi ($74+) dari churn. |


## 4. Jembatan ke Data Science Track
Eksplorasi ini membuktikan fitur-fitur yang paling berpengaruh terhadap retensi:
1. `contract` (Month-to-month)
2. `tenure` (Non-linear hazard curve)
3. `monthly_charges` / `total_charges`
4. Interaksi `internet_service` (Fiber optic) dan `tech_support`
5. `payment_method` (Electronic check)

Fitur-fitur ini akan ditransformasikan dalam *Science Track* (`notebooks/02_prediction/`) ke dalam pipeline machine learning tanpa kebocoran data (*zero data leakage*).
